In [ ]:
import os
import datetime

from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain.schema import HumanMessage, SystemMessage

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.errors import NodeInterrupt

from typing import TypedDict, Optional

In [ ]:
load_dotenv()

GroqApiKey = os.getenv("GroqAPI")
ModelName = "llama-3.3-70b-versatile"
LogDirectory = "hitl_logs"

if not GroqApiKey:
    raise ValueError("GroqAPI not found in .env")

print(f"Model: {ModelName}")
print(f"Log Dir: {LogDirectory}")

In [ ]:
class AgentState(TypedDict):
    
    Task: str           
    ActionPlan: str            

    HumanDecision: str           
    HumanEdit: Optional[str]

    ExecutionResult: str            

In [ ]:
LLM = ChatGroq(
    api_key = GroqApiKey,
    model = ModelName,
    temperature = 0.4,
)

In [ ]:
def plannerNode(State: AgentState) -> dict:
    
    Task = State["Task"]

    print("\n[Planner] Generating action plan")

    Messages = [
        SystemMessage(content=(
            "You are a professional task planner. Given a high-stakes task, produce a clear, numbered action plan. Each step should be specific and actionable. "
            "Format: Step 1: ... / Step 2: ... etc. "
            "Keep it concise, 4 to 6 steps maximum."
        )),
        HumanMessage(content=f"Task: {Task}")
    ]

    Response   = LLM.invoke(Messages)
    ActionPlan = Response.content

    print("[Planner] Plan generated.")
    return {"ActionPlan": ActionPlan}

In [ ]:
def humanApprovalNode(State: AgentState) -> dict:
    
    if not State.get("HumanDecision"):
        ActionPlan = State["ActionPlan"]

        print("\n\n")
        print("  Action Plan: Human Review\n")
        print(ActionPlan)
        print("\n")

        raise NodeInterrupt(
            "Plan ready for human review. "
            "Call resume_with_decision() to continue."
        )
        
    print(f"[HumanApproval] Decision received: {State['HumanDecision']}")
    return {}

In [ ]:
def executorNode(State: AgentState) -> dict:

    PlanToExecute = State.get("HumanEdit") or State["ActionPlan"]
    Task = State["Task"]

    print("\n[Executor] Running approved plan")

    Messages = [
        SystemMessage(content=(
            "You are an AI executor. You have been given an approved action plan. "
            "Simulate carrying out each step and provide a realistic completion summary. "
            "Be specific about what was done at each step."
        )),
        HumanMessage(content=(
            f"Original Task: {Task}\n\n"
            f"Approved Plan:\n{PlanToExecute}\n\n"
            "Please execute this plan and provide a summary of what was completed."
        ))
    ]

    Response = LLM.invoke(Messages)
    ExecutionResult = Response.content

    print("[Executor] Execution complete.")
    return {"ExecutionResult": ExecutionResult}

In [ ]:
def routeAfterApproval(State: AgentState) -> str:
   
    Decision = State.get("HumanDecision", "").lower().strip()

    if Decision == "reject":
        print("[Router] Plan rejected. Routing to END.")
        return END

    # yes or edit both proceed to execution
    print("[Router] Plan approved. Routing to executor.")
    return "executor"

In [ ]:
def buildGraph():
   
    Builder = StateGraph(AgentState)
    Memory = MemorySaver()  

    Builder.add_node("planner", plannerNode)
    Builder.add_node("human_approval", humanApprovalNode)
    Builder.add_node("executor", executorNode)

    Builder.add_edge(START, "planner")
    Builder.add_edge("planner", "human_approval")
 
    Builder.add_conditional_edges( "human_approval", routeAfterApproval, {"executor": "executor", END: END} )

    Builder.add_edge("executor", END)

    Graph = Builder.compile(checkpointer=Memory)

    return Graph

Graph = buildGraph()

In [ ]:
def createAuditLog(LogDir: str) -> str:
    
    os.makedirs(LogDir, exist_ok=True)
    Timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    LogFilePath = os.path.join(LogDir, f"hitl_audit_{Timestamp}.txt")

    with open(LogFilePath, "w", encoding="utf-8") as F:
        F.write("\n")
        F.write("Human In The Loop Log\n")
        F.write(f"Session: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        F.write(f"Model: {ModelName}\n")
        F.write("\n\n")

    print(f"Audit log created: {LogFilePath}")
    return LogFilePath

def writeAuditEntry(LogFilePath: str, Section: str, Content: str):

    Timestamp = datetime.datetime.now().strftime("%H:%M:%S")

    with open(LogFilePath, "a", encoding="utf-8") as F:
        F.write(f"[{Timestamp}] {Section}\n")
        F.write("\n")
        F.write(Content.strip() + "\n\n")

In [ ]:
def collectHumanDecision(Graph, Config: dict, CurrentState: dict) -> tuple:

    while True:
        print("\nOptions: yes | edit | reject")
        RawInput = input("Your decision: ").strip().lower()

        if RawInput in ("yes", "edit", "reject"):
            Decision = RawInput
            break
        else:
            print(f"  Invalid input '{RawInput}'. Please type yes, edit, or reject.")

    EditedPlan = None

    if Decision == "edit":
        print("\nEnter your revised plan.")
        print("(Type END on a new line when finished)")

        Lines = []
        while True:
            Line = input()
            if Line.strip().upper() == "END":
                break
            Lines.append(Line)

        EditedPlan = "\n".join(Lines).strip()

        if not EditedPlan:
            print("No edit entered, treating as 'yes'.")
            Decision = "yes"
            EditedPlan = None

    StateUpdate = {"HumanDecision": Decision}

    if EditedPlan:
        StateUpdate["HumanEdit"] = EditedPlan

    Graph.update_state( config = Config, values = StateUpdate, as_node = "human_approval", )

    print(f"\n[update_state] Injected into checkpoint: {list(StateUpdate.keys())}")

    return Decision, EditedPlan

In [ ]:
def runAgent(Task: str):

    LogFilePath = createAuditLog(LogDirectory)

    Timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    ThreadId = f"hitl-thread-{Timestamp}"
    Config = {"configurable": {"thread_id": ThreadId}}

    writeAuditEntry(LogFilePath, "Task", Task)

    print("\n")
    print("Human Approved Agent")
    print("\n")
    print(f"Task: {Task}")

    print("\n\tPhase 1: Planning")

    FirstResult = Graph.invoke( {"Task": Task}, config = Config )

    ActionPlan = FirstResult.get("ActionPlan", "")

    writeAuditEntry(LogFilePath, "Original Plan", ActionPlan)

    print("\n\tPhase 2: Human Review")

    Decision, EditedPlan = collectHumanDecision(Graph, Config, FirstResult)
    
    writeAuditEntry(LogFilePath, f"Human Decesion", Decision.upper())

    if EditedPlan:
        writeAuditEntry(LogFilePath, "Human edit", EditedPlan)

    print("\n\tPhase 3: Resuming Graph")

    FinalResult = Graph.invoke(None, config=Config)

    ExecutionResult = FinalResult.get("ExecutionResult", "")

    if Decision == "reject":
        print("\n\n")
        print("Result: Plan rejected. No action taken.")
    
        writeAuditEntry(LogFilePath, "Execution Result", "Reject, no action taken.")

    else:
        print("\n\n")
        print("Execution")
        print(ExecutionResult)
        writeAuditEntry(LogFilePath, "Execution Result", ExecutionResult)

    print(f"\nAudit log saved to: {LogFilePath}")
    return FinalResult

In [ ]:
UserTask = "Draft and send a project update email to all stakeholders informing them of a two-week delay in the product launch"

Result = runAgent(UserTask)

In [ ]:
def runInteractiveSession():
   
    print("\n\n")
    print("HITL Agent: Interactive Session")
    print("Type 'quit' to exit\n")

    while True:
        try:
            Task = input("\nEnter task: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSession ended.")
            break

        if Task.lower() in ("quit", "exit", "q"):
            print("Goodbye!")
            break

        if not Task:
            print("(Empty input — please describe a task.)")
            continue

        try:
            runAgent(Task)
        except Exception as Error:
            print(f"\n[Error]: {Error}")
            print("Please try again with a different task.")

runInteractiveSession()

In [ ]:
def showLatestAuditLog(LogDir: str):
    
    if not os.path.exists(LogDir):
        print("No logs found yet.")
        return

    LogFiles = sorted( [F for F in os.listdir(LogDir) if F.startswith("hitl_audit_")] )
    
    if not LogFiles:
        print("No audit logs found.")
        return

    LatestLog = os.path.join(LogDir, LogFiles[-1])
    print(f"Reading: {LatestLog}\n")

    with open(LatestLog, "r", encoding="utf-8") as F:
        print(F.read())

showLatestAuditLog(LogDirectory)